# EasyRAG Build Index Walkthrough

This notebook shows what it means to build an index in EasyRAG. The structure follows the same teaching logic used by from-scratch RAG tutorials: first create a tiny corpus, then inspect how it becomes documents and chunks, then build the searchable workspace that retrieval will use later.

## What you will do

- create a temporary repository-shaped corpus
- load repository documents through the canonical EasyRAG entry point
- inspect chunk boundaries and chunking strategies
- rebuild an EasyRAG workspace index without API keys
- verify the built index with a retrieval query

## Why this notebook starts with indexing rather than answering

Retrieval quality depends on index quality. If you do not understand what was loaded, how it was chunked, and where it was stored, later retrieval behavior will feel mysterious.


## Step 1: Import the indexing and query helpers

We will use the public `EasyRAG` API for loading and querying, and a small set of indexing helpers for chunk inspection and workspace rebuilds. `run_sync` is still useful because the orchestrator lifecycle is async even inside a notebook.


In [ ]:
import json
import os
import tempfile
from pathlib import Path

from easyrag import EasyRAG, QueryParam
from easyrag.rag.indexing import ChunkingConfig, chunk_documents, rebuild_document_index
from easyrag.support.async_utils import run_sync


This cell should only import symbols. The important thing to notice is the separation of roles: `EasyRAG` is the high-level entry point, while `chunk_documents()` and `rebuild_document_index()` expose the indexing mechanics more directly.


## Step 2: Create a tiny repository-shaped corpus

Instead of indexing the full EasyRAG repository, we create a small temporary repo with a `README.md` and a few files under `docs/`. That keeps the example deterministic and makes it obvious why each document was loaded.


In [ ]:
repo_temp_dir = tempfile.TemporaryDirectory()
repo_root = Path(repo_temp_dir.name) / "demo_repo"
docs_dir = repo_root / "docs"
docs_dir.mkdir(parents=True)

(repo_root / "README.md").write_text(
    "# Demo Repo\nThis repository explains EasyRAG indexing.\n",
    encoding="utf-8",
)
(docs_dir / "architecture.md").write_text(
    "# Architecture\nEasyRAG organizes indexing as loading, chunking, and retrieval.\n## Chunking\nStructured chunks keep section boundaries.\n",
    encoding="utf-8",
)
(docs_dir / "notes.txt").write_text(
    "Semantic chunking works well for plain text notes about embeddings and retrieval.\n",
    encoding="utf-8",
)
(docs_dir / "operations.md").write_text(
    "# Operations\nThe build script can rebuild, update, or delete index entries.\n",
    encoding="utf-8",
)

print(json.dumps(sorted(str(path.relative_to(repo_root)) for path in repo_root.rglob("*") if path.is_file()), indent=2))


You should see a small file list rooted at `demo_repo/`. This is the corpus we will index. Because the directory layout looks like a repository, the same loader logic used on real docs can operate on it without any special casing.


## Step 3: Load repository documents

`EasyRAG.load_repo_documents()` is the teaching-friendly entry point for repository discovery. It applies the repo loader rules and returns canonical `Document` objects with metadata such as title, path, and `doc_id`.


In [ ]:
documents = EasyRAG.load_repo_documents(repo_root)

document_preview = [
    {
        "title": str(document.metadata.get("title")),
        "relative_path": str(document.metadata.get("relative_path")),
        "doc_id": str(document.metadata.get("doc_id")),
        "source_type": str(document.metadata.get("source_type")),
    }
    for document in documents
]

print(json.dumps(document_preview, indent=2))


You should see one record per loaded source file. The key observation is that indexing does not begin with anonymous text blobs. It begins with documents that already carry stable metadata, and that metadata will later show up in citations and storage records.


## Step 4: Inspect chunk boundaries before building the workspace

Chunking is where raw documents become retrieval units. We inspect chunks before writing any index so you can see which strategy EasyRAG selected and what metadata each chunk carries.


In [ ]:
chunks = chunk_documents(documents, config=ChunkingConfig())

chunk_preview = [
    {
        "doc_id": str(chunk.metadata.get("doc_id")),
        "title": str(chunk.metadata.get("title")),
        "chunk_id": int(chunk.metadata.get("chunk_id", 0)),
        "chunk_strategy": str(chunk.metadata.get("chunk_strategy")),
        "overlap_policy": str(chunk.metadata.get("overlap_policy")),
        "snippet": chunk.page_content[:90],
    }
    for chunk in chunks
]

print(json.dumps(chunk_preview, indent=2))


This output should show a mix of strategies. Markdown files usually become `structured` chunks, while plain text files often fall back to `sliding_window` when no semantic embedding function is available. That is a useful reminder that chunking is adaptive, not one-size-fits-all.


## Step 5: Configure an isolated build workspace

The maintenance helpers use environment-backed paths. To keep this notebook self-contained, we point those paths at a temporary data directory instead of writing into the real project workspace. We also leave `OPENAI_API_KEY` empty on purpose so the example remains zero-key runnable.


In [ ]:
ENV_KEYS = [
    "EASYRAG_DATA_DIR",
    "EASYRAG_INDEX_PATH",
    "EASYRAG_WORKING_DIR",
    "EASYRAG_WORKSPACE",
    "OPENAI_API_KEY",
]

env_backup = {key: os.environ.get(key) for key in ENV_KEYS}

data_dir = Path(repo_temp_dir.name) / ".easyrag"
index_path = data_dir / "rag_index.json"
working_dir = Path(repo_temp_dir.name) / "workspaces"
workspace = "build-index-demo"

os.environ["EASYRAG_DATA_DIR"] = str(data_dir)
os.environ["EASYRAG_INDEX_PATH"] = str(index_path)
os.environ["EASYRAG_WORKING_DIR"] = str(working_dir)
os.environ["EASYRAG_WORKSPACE"] = workspace
os.environ["OPENAI_API_KEY"] = ""

print(json.dumps({
    "data_dir": str(data_dir),
    "index_path": str(index_path),
    "working_dir": str(working_dir),
    "workspace": workspace,
    "openai_api_key_configured": bool(os.environ["OPENAI_API_KEY"]),
}, indent=2))


You should see all persistence paths pointing inside the temporary directory, and `openai_api_key_configured` should be `false`. In this configuration, EasyRAG will still build the workspace and later query it through the fallback token backend.


## Step 6: Rebuild the index from the repository root

`rebuild_document_index()` is the most direct library-level equivalent of the CLI build script. It discovers documents from a repo root, writes the EasyRAG workspace, and also emits the lightweight compatibility snapshot at `EASYRAG_INDEX_PATH`.


In [ ]:
workspace_path = rebuild_document_index(repo_root)

print(json.dumps({
    "workspace_path": str(workspace_path),
    "legacy_snapshot_exists": index_path.exists(),
}, indent=2))


This cell should return the built workspace path and confirm that the legacy snapshot file exists. The important conceptual point is that one rebuild operation creates both the richer EasyRAG workspace and the simpler compatibility snapshot.


## Step 7: Inspect what the build produced on disk

A build is easier to trust when you can inspect its artifacts. We will look at the generated workspace files and sample one item from the compatibility snapshot.


In [ ]:
workspace_files = sorted(
    str(path.relative_to(workspace_path))
    for path in workspace_path.rglob("*")
    if path.is_file()
)
legacy_snapshot = json.loads(index_path.read_text(encoding="utf-8"))

print(json.dumps({
    "workspace_files": workspace_files,
    "legacy_snapshot_items": len(legacy_snapshot),
    "first_snapshot_metadata": legacy_snapshot[0]["metadata"],
}, indent=2))


You should see storage files for documents, chunks, vectors, graph state, and document status, plus the legacy JSON snapshot. This is what index building means in concrete terms: the source repo has been turned into persistent retrieval artifacts.


## Step 8: Open the built workspace and inspect aggregate stats

Now that the index exists, we reopen it through `EasyRAG` and ask for aggregate statistics. This gives a compact summary of how many documents, chunks, vectors, and graph records were created.


In [ ]:
rag = EasyRAG(working_dir=working_dir, workspace=workspace)
run_sync(rag.initialize_storages())

stats = run_sync(rag.get_stats())
print(json.dumps(stats, indent=2))


Notice the `chunk_strategy_counts` and `vector_backend` fields. In this zero-key example, the backend should be `fallback_token`, which is exactly what we want for a notebook that stays runnable without external model configuration.


## Step 9: Verify the index with a retrieval query

The build is not very useful unless retrieval can consume it. We run one narrow query with query rewriting and MQE disabled so the result reflects the index contents as directly as possible.


In [ ]:
result = run_sync(
    rag.aquery(
        "What keeps section boundaries?",
        QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False),
    )
)

print(json.dumps({
    "citations": result.citations,
    "metadata": result.metadata,
}, indent=2))


You should see a citation pointing at the `Chunking` section from `docs/architecture.md`. That is the practical proof that the index build succeeded: the retrieval layer can now surface grounded snippets from the stored corpus.


## Step 10: Clean up the notebook workspace

We close the storages, restore the previous environment variables, and remove the temporary repository. This mirrors good lifecycle hygiene and avoids leaking notebook state into later cells.


In [ ]:
run_sync(rag.finalize_storages())

for key, value in env_backup.items():
    if value is None:
        os.environ.pop(key, None)
    else:
        os.environ[key] = value

repo_temp_dir.cleanup()

print("Closed the workspace, restored environment variables, and removed the temporary repository.")


After this cleanup cell, the demo index is gone. That is expected. The notebook's job is to teach the build process, not to leave behind persistent state in your real project directories.

## Step 11: Map the notebook flow to the real CLI

The last cell does not run the actual script. It prints the commands you would use when you are ready to rebuild the real repository index from the command line.


In [ ]:
cli_templates = {
    "full_rebuild": "python scripts/build_index.py --action rebuild",
    "targeted_update": "python scripts/build_index.py --action update --doc-id <doc-id>",
    "targeted_delete": "python scripts/build_index.py --action delete --doc-id <doc-id>",
}

print(json.dumps(cli_templates, indent=2))


These commands are the operational version of what you just did manually. The notebook taught the concepts in slow motion; `scripts/build_index.py` packages the same lifecycle into a repeatable repo-level workflow.

## Next steps

- read [docs/01-rag-basics.md](../docs/01-rag-basics.md) again now that you have seen a real build
- continue to [docs/03-indexing-overview.md](../docs/03-indexing-overview.md) for a deeper explanation of ingestion internals
- compare this notebook with [00_quickstart.ipynb](00_quickstart.ipynb) to see how build-time and query-time concerns connect

If you can explain why the loaded documents became the observed chunks and why those chunks later produced the final citation, you already understand the first indexing loop in EasyRAG.
